# Competencia: pregúntale a Wikipedia

**Sesión 9 · IA Intermedio · EstudIA 2026**

Construye un RAG que responda **200 preguntas que personas en Colombia le hicieron a Google**, buscando entre
**6.763 párrafos de Wikipedia** en español.

**Cómo se califica.** Para cada pregunta envías los párrafos que encontró tu buscador y la respuesta de tu modelo. El
puntaje es el promedio de dos partes:

- **Búsqueda:** ¿en qué puesto envías el párrafo correcto? 1 punto si es el primero, 1/2 si es el segundo, 1/3 si es
  el tercero… hasta el quinto; 0 si no está entre tus 5 primeros. Ojo: hay varios párrafos de cada artículo.
- **Respuesta:** un modelo juez (Jev) compara tu respuesta con la respuesta esperada, que escribimos a mano a partir
  del párrafo. Cuenta si dice lo mismo, aunque use otras palabras. Las respuestas de más de 300 caracteres se cortan.

La tabla en vivo usa la mitad de las preguntas; la otra mitad decide el resultado final. Tienes **20 envíos al día**:
mide tus cambios antes con el **conjunto de validación**, que trae el párrafo correcto (y, en 40 preguntas, la
respuesta esperada).

**Reglas.** Usa los modelos de la plataforma (cualquiera de los cuatro) y el buscador que quieras. Las respuestas
tienen que salir de tu sistema: no las escribas ni las busques a mano.

El notebook tal como viene (BM25 y el modelo más flojo de los cuatro, con instrucciones simples) llega a
**16,9 %** en la tabla.

**Antes de empezar.** *Archivo › Guardar una copia en Drive*. No necesitas GPU.

In [ ]:
# Preparación: librerías y tu clave. Ejecútala una vez (instalar tarda un momento).
!pip -q install "langchain-openai>=1.8" "langchain-qdrant>=1.2" "langchain-community>=0.4" "langchain-huggingface>=1.3" rank_bm25
import os, re, csv, json, time, getpass, urllib.request, urllib.error, warnings
warnings.filterwarnings("ignore")
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s09")
PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import openai

_CLAVE = os.environ.get("ESTUDIA_CLAVE") or getpass.getpass(
    "Pega tu clave (plataforma › Tu clave para los modelos de lenguaje › Copiar) y oprime Enter: ").strip()
cliente = openai.OpenAI(base_url=f"{PLATAFORMA}/v1", api_key=_CLAVE, max_retries=3)
try:
    MODELOS = [m.id for m in cliente.models.list().data if getattr(m, "tipo", "chat") == "chat"]
    print("Listo. Modelos de lenguaje disponibles:", ", ".join(MODELOS))
except openai.AuthenticationError:
    print("La clave no es válida. Cópiala otra vez de la plataforma y vuelve a ejecutar esta celda.")
except openai.APIConnectionError:
    print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")

In [ ]:
# @title Funciones auxiliares: modelos de lenguaje, embeddings, Qdrant, buscar, medir, juez, registro y enviar
def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", os.path.basename(nombre))
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

import torch
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.retrievers import BM25Retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
torch.set_num_threads(max(1, os.cpu_count() or 1))

def modelo_de_lenguaje(nombre, temperatura=0, max_tokens=400, **extra):
    """Un modelo de lenguaje de la plataforma, para usarlo con LangChain."""
    return ChatOpenAI(model=nombre, base_url=f"{PLATAFORMA}/v1", api_key=_CLAVE, temperature=temperatura,
                      max_tokens=max_tokens, max_retries=6, timeout=120, **extra)

def preguntar(modelo, pregunta, sistema=None, mostrar=True, **opciones):
    """Una pregunta a un modelo, sin contexto. Devuelve la respuesta y muestra el tiempo y el costo."""
    mensajes = ([{"role": "system", "content": sistema}] if sistema else []) + [{"role": "user", "content": pregunta}]
    inicio = time.time()
    try:
        r = cliente.chat.completions.create(model=modelo, messages=mensajes, **opciones)
    except openai.APIStatusError as e:
        print("No se pudo:", (e.body or {}).get("message", e))
        return ""
    texto = r.choices[0].message.content or ""
    if mostrar:
        u = r.usage.model_dump()
        segundos = f"{time.time() - inicio:.1f}".replace(".", ",")
        costo = f"{u.get('costo_usd', 0):.6f}".replace(".", ",")
        print(f"{texto}\n  ({modelo} · {segundos} s · {u['prompt_tokens']} tokens de entrada y "
              f"{u['completion_tokens']} de salida · US$ {costo})")
    return texto

def cuanto_llevo():
    """Lo que llevas gastado hoy en la API de modelos de lenguaje."""
    req = urllib.request.Request(f"{PLATAFORMA}/v1/usage", headers={"Authorization": f"Bearer {_CLAVE}",
                                                                    "User-Agent": "EstudIA-notebook/1.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        u = json.loads(r.read())
    hoy, limite = f"{u['hoy_usd']:.4f}".replace(".", ","), f"{u['limite_diario_usd']:.2f}".replace(".", ",")
    print(f"Hoy llevas US$ {hoy} de US$ {limite} ({u['solicitudes_hoy']} solicitudes).")

_CON_PREFIJOS = []

class ConPrefijos(Embeddings):
    """Embeddings de la familia e5: “query: ” antes de cada pregunta y “passage: ” antes de cada texto (sesión 6).
    Guarda los vectores de las preguntas: preparar() los calcula todos de una vez."""
    def __init__(self, base, consulta="query: ", documento="passage: "):
        self.base, self.consulta, self.documento, self._cache = base, consulta, documento, {}
        _CON_PREFIJOS.append(self)
    def embed_documents(self, textos):
        return self.base.embed_documents([self.documento + t for t in textos])
    def embed_query(self, texto):
        if texto not in self._cache:
            self._cache[texto] = self.base.embed_query(self.consulta + texto)
        return self._cache[texto]
    def preparar(self, preguntas):
        nuevas = [p for p in dict.fromkeys(preguntas) if p not in self._cache]
        if nuevas:
            self._cache.update(zip(nuevas, self.base.embed_documents([self.consulta + p for p in nuevas])))

def embeddings_locales(nombre="intfloat/multilingual-e5-small"):
    """Un modelo de embeddings de Hugging Face que corre en Colab (sin GPU también)."""
    from langchain_huggingface import HuggingFaceEmbeddings
    return ConPrefijos(HuggingFaceEmbeddings(model_name=nombre, encode_kwargs={"normalize_embeddings": True,
                                                                              "batch_size": 32}))

def embeddings_api(nombre="intfloat/multilingual-e5-large"):
    """Un modelo de embeddings más grande, a través de la plataforma (cuesta, aunque muy poco)."""
    return ConPrefijos(OpenAIEmbeddings(model=nombre, base_url=f"{PLATAFORMA}/v1", api_key=_CLAVE,
                                        check_embedding_ctx_length=False, chunk_size=128, max_retries=6))

def coleccion(documentos, embeddings, nombre="documentos"):
    """Una colección de Qdrant en la memoria de Colab con los vectores de los documentos."""
    inicio = time.time()
    c = QdrantVectorStore.from_documents(documentos, embeddings, location=":memory:", collection_name=nombre)
    print(f"{miles(len(documentos))} documentos en Qdrant en {time.time() - inicio:.0f} s.")
    return c

def buscar_hibrido(listas, k=5, c=60):
    """Combina varias listas de resultados (de buscadores distintos): gana lo que sale arriba en varias."""
    puntaje, docs = {}, {}
    for lista in listas:
        for puesto, d in enumerate(lista):
            clave = d.metadata.get("id", d.page_content[:80])
            puntaje[clave] = puntaje.get(clave, 0) + 1 / (c + puesto + 1)
            docs[clave] = d
    return [docs[i] for i in sorted(puntaje, key=lambda i: -puntaje[i])[:k]]

def preparar_preguntas(preguntas):
    for e in _CON_PREFIJOS:
        e.preparar(list(preguntas))

def medir_busqueda(buscar, preguntas, correctos, ks=(1, 3, 5), clave="id"):
    """En qué fracción de las preguntas el documento correcto sale entre los primeros k resultados."""
    preparar_preguntas(preguntas)
    aciertos = {k: 0 for k in ks}
    for p, ok in zip(preguntas, correctos):
        ids = [d.metadata.get(clave) for d in buscar(p)]
        ok = ok if isinstance(ok, (set, list, tuple)) else {ok}
        for k in ks:
            aciertos[k] += any(i in ok for i in ids[:k])
    return {f"en los primeros {k}": aciertos[k] / len(preguntas) for k in ks}

def puntos_busqueda(buscar, preguntas, correctos, k=5):
    """Como la plataforma: 1 punto si el correcto sale primero, 1/2 si sale segundo… 0 si no está entre los k primeros."""
    preparar_preguntas(preguntas)
    total = 0.0
    for p, ok in zip(preguntas, correctos):
        ids = [d.metadata.get("id") for d in buscar(p)][:k]
        total += 1 / (ids.index(ok) + 1) if ok in ids else 0.0
    return total / len(preguntas)

def responder_lote(cadena, buscar, preguntas, formato, k=5, simultaneas=4):
    """Busca y responde muchas preguntas a la vez. Devuelve (respuestas, documentos de cada pregunta)."""
    preparar_preguntas(preguntas)
    docs = [buscar(p)[:k] for p in preguntas]
    salidas = cadena.batch([{"contexto": formato(d), "pregunta": p} for d, p in zip(docs, preguntas)],
                           config={"max_concurrency": simultaneas}, return_exceptions=True)
    errores = [s for s in salidas if not isinstance(s, str)]
    if errores:
        print(f"{len(errores)} preguntas no se pudieron responder; por ejemplo: {str(errores[0])[:200]}")
    return [" ".join(s.split()) if isinstance(s, str) else "" for s in salidas], docs

JUEZ = ("Compara la respuesta de un sistema con la respuesta correcta a una pregunta. Escribe solo SI si dice lo "
        "mismo en lo esencial (aunque use otras palabras o agregue detalles) o NO si dice otra cosa, si le falta el "
        "dato principal o si dice que no sabe.")

def juez(preguntas, esperadas, respuestas, modelo="mistralai/mistral-small-3.2-24b-instruct"):
    """Un modelo de lenguaje como juez: ¿cada respuesta dice lo mismo que la esperada? Aproximado (y cuesta)."""
    cadena = (ChatPromptTemplate.from_messages([("system", JUEZ), ("human", "Pregunta: {p}\nRespuesta correcta: {e}\n"
                                                                          "Respuesta del sistema: {a}")])
              | modelo_de_lenguaje(modelo, max_tokens=5) | StrOutputParser())
    salidas = cadena.batch([{"p": p, "e": e, "a": a or "(sin respuesta)"} for p, e, a in zip(preguntas, esperadas, respuestas)],
                           config={"max_concurrency": 4}, return_exceptions=True)
    errores = [s for s in salidas if not isinstance(s, str)]
    if errores:
        print(f"El juez no pudo revisar {len(errores)} respuestas (cuentan como no aceptadas); por ejemplo: "
              f"{str(errores[0])[:200]}")
    return [isinstance(s, str) and s.strip().upper().startswith("S") for s in salidas]

from IPython.display import display
REGISTRO = "experimentos.csv"

def registrar(**datos):
    """Agrega una fila a tu registro de experimentos (el archivo experimentos.csv): qué probaste y qué salió."""
    fila = pd.DataFrame([{"hora": time.strftime("%H:%M"), **datos}])
    tabla = pd.concat([pd.read_csv(REGISTRO), fila], ignore_index=True) if os.path.exists(REGISTRO) else fila
    tabla.to_csv(REGISTRO, index=False)

def ver_registro():
    """Muestra el registro de experimentos, el más reciente abajo."""
    if not os.path.exists(REGISTRO):
        print("Todavía no hay experimentos en el registro.")
        return
    tabla = pd.read_csv(REGISTRO)
    display(tabla.style.format(precision=1, decimal=",", thousands=".", na_rep="").hide(axis="index"))

def enviar(ids, pasajes, respuestas, nota=""):
    """Envía tus respuestas a la plataforma del curso y muestra tu puntaje en la tabla.

    ids: los ids de las preguntas de prueba. pasajes: para cada pregunta, la lista de ids de los párrafos que encontró
    tu buscador (cuentan los 5 primeros). respuestas: la respuesta de tu RAG a cada pregunta.
    nota: un texto corto para recordar qué probaste (opcional)."""
    cuerpo = json.dumps({"ids": [int(i) for i in ids], "pasajes": [[str(x) for x in p] for p in pasajes],
                         "respuestas": [str(r) for r in respuestas], "nota": str(nota)[:80]}).encode()
    req = urllib.request.Request(f"{PLATAFORMA}/api/competencia/s09/envios", data=cuerpo, method="POST",
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {_CLAVE}",
                                          "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=300) as r:  # el juez tarda un poco
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        print("No se pudo enviar:", res.get("error", f"error {e.code}"))
        return None
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return None
    print(f"Envío {res['envio']} recibido. Puntaje RAG en la tabla: {pct(res['puntaje'])} "
          f"(búsqueda {pct(res['busqueda'])}, respuestas {pct(res['respuestas'])}).")
    print(f"Tu mejor resultado: {pct(res['mejor'])} · puesto {res['puesto']} de {res['participantes']} ({res['alias']}). "
          f"Te quedan {res.get('envios_restantes_hoy', '?')} envíos hoy.")
    return res

---
## 1. Los datos

Las preguntas vienen de MessIRve (búsquedas reales hechas en Colombia); los párrafos, de Wikipedia en español. Cada
párrafo tiene un id, como `"111262#0"`, y el título del artículo de donde salió.

In [ ]:
parrafos = [json.loads(linea) for linea in open(descargar("pasajes.jsonl"), encoding="utf-8")]
documentos = [Document(page_content=p["texto"], metadata={"id": p["id"], "titulo": p["titulo"], "texto": p["texto"]})
              for p in parrafos]
prueba = pd.read_csv(descargar("preguntas_prueba.csv"))
validacion = pd.read_csv(descargar("validacion.csv")).fillna("")
con_respuesta = validacion[validacion["respuesta_esperada"] != ""]
print(f"{miles(len(documentos))} párrafos · {len(prueba)} preguntas de prueba · {len(validacion)} de validación "
      f"({len(con_respuesta)} con respuesta esperada)")
display(validacion.head(3))
print("Un párrafo:", parrafos[0]["titulo"], "→", parrafos[0]["texto"][:200], "…")

---
## 2. El buscador: BM25

BM25 es una bolsa de palabras (sesión 5): da más puntos a los párrafos que comparten palabras raras con la pregunta.
Medimos en cuántas preguntas de validación encuentra el párrafo correcto, y los puntos de búsqueda que le daría la
plataforma (según el puesto del correcto).

In [ ]:
bm25 = BM25Retriever.from_documents(documentos, k=5)

def buscar(pregunta):
    """Los párrafos para una pregunta, el mejor primero (cuentan los 5 primeros)."""
    return bm25.invoke(pregunta)

busqueda = medir_busqueda(buscar, validacion["pregunta"], validacion["pasaje"])
print({k: pct(v) for k, v in busqueda.items()})
print("Puntos de búsqueda:", pct(puntos_busqueda(buscar, validacion["pregunta"], validacion["pasaje"])))

---
## 3. El modelo: responder con el contexto

In [ ]:
MODELO = "google/gemini-2.5-flash-lite"   # el más flojo de los cuatro: prueba los otros (MODELOS)
INSTRUCCIONES = "Responde la pregunta usando el contexto."

plantilla = ChatPromptTemplate.from_messages([("system", INSTRUCCIONES),
                                              ("human", "Contexto:\n{contexto}\n\nPregunta: {pregunta}")])
cadena = plantilla | modelo_de_lenguaje(MODELO) | StrOutputParser()

def formato(docs):
    """Cómo ve el modelo los párrafos: id, título y texto."""
    return "\n\n".join(f"[{d.metadata['id']}] {d.metadata['titulo']}: {d.metadata['texto']}" for d in docs)

for _, fila in con_respuesta.head(3).iterrows():
    r = cadena.invoke({"contexto": formato(buscar(fila["pregunta"])), "pregunta": fila["pregunta"]})
    print(f"{fila['pregunta']}\n  esperada: {fila['respuesta_esperada']}\n  tu RAG:   {r}\n")

---
## 4. Validar antes de enviar

Las 40 preguntas de validación con respuesta esperada: tu RAG las responde y un modelo juez (aproximado, no es Jev)
dice cuántas acepta. Así comparas cambios sin gastar envíos.

In [ ]:
respuestas_val, docs_val = responder_lote(cadena, buscar, list(con_respuesta["pregunta"]), formato)
ok = juez(con_respuesta["pregunta"], con_respuesta["respuesta_esperada"], respuestas_val)
busq = puntos_busqueda(buscar, validacion["pregunta"], validacion["pasaje"])
print(f"Validación: búsqueda {pct(busq)} · respuestas que acepta el juez {pct(sum(ok) / len(ok))} · "
      f"puntaje aproximado {pct((busq + sum(ok) / len(ok)) / 2)}")
registrar(buscador="BM25 tal cual", modelo=MODELO, instrucciones=INSTRUCCIONES[:60], busqueda_val=100 * busq,
          respuestas_val=100 * sum(ok) / len(ok))
cuanto_llevo()

---
## 5. Enviar

Responde las 200 preguntas de prueba y envía los párrafos y las respuestas. Tarda un par de minutos.

In [ ]:
respuestas, docs = responder_lote(cadena, buscar, list(prueba["pregunta"]), formato)
pasajes = [[d.metadata["id"] for d in ds] for ds in docs]
res = enviar(prueba["id"], pasajes, respuestas, nota=f"BM25 · {MODELO.split('/')[-1]}")

---
## 6. Pistas

Ábrelas cuando quieras. Prueba un cambio a la vez, mídelo en validación y anótalo con `registrar(...)`.

<details><summary>Pista 1: busca por significado</summary>

BM25 solo encuentra palabras iguales. Con embeddings (sesión 6) y Qdrant, “dónde queda” encuentra “está ubicado”:

```python
EMB = embeddings_locales("intfloat/multilingual-e5-small")      # en Colab, sin costo; o embeddings_api(), más grande
parrafos_qdrant = coleccion(documentos, EMB, "parrafos")
def buscar(pregunta):
    return parrafos_qdrant.similarity_search(pregunta, k=5)
```
Los modelos e5 esperan “query: ” y “passage: ” antes de cada texto; `ConPrefijos` ya los pone.
</details>

<details><summary>Pista 2: el título dice mucho</summary>

Muchos párrafos de Wikipedia no repiten el nombre de su artículo: el párrafo de “Villavicencio” puede decir solo “es un
municipio colombiano…”. Pon el título al principio del texto que conviertes en vector (y del que lee BM25).
</details>

<details><summary>Pista 3: un BM25 más listo</summary>

`BM25Retriever.from_documents(..., preprocess_func=...)` decide cómo se parten los textos en palabras. Por defecto
distingue mayúsculas y tildes, y cuenta “de”, “la” y “que”. Prueba una función que pase a minúsculas, quite las
tildes y descarte las palabras vacías.
</details>

<details><summary>Pista 4: lo mejor de dos buscadores</summary>

BM25 acierta con nombres raros; los embeddings, con preguntas dichas de otra forma. `buscar_hibrido([lista1, lista2])`
combina sus resultados: gana lo que sale arriba en las dos listas.
</details>

<details><summary>Pista 5: respuestas como las del juez</summary>

El juez compara con respuestas cortas, de una frase, con el dato que pide la pregunta. Si tu modelo responde con tres
párrafos, rodeos o “según el contexto…”, pierde puntos (y después de 300 caracteres se corta). Pídeselo en las
instrucciones.
</details>

<details><summary>Pista 6: cambia de modelo</summary>

Empiezas con el más flojo de los cuatro. Compáralos en validación: aciertos, tiempo y costo (`cuanto_llevo()`). A los
que razonan (Gemini, GLM y GPT) puedes pedirles más o menos esfuerzo: `modelo_de_lenguaje(m,
reasoning_effort="medium")`.
</details>

<details><summary>Pista 7: cuánto contexto</summary>

Le das al modelo 5 párrafos. Con menos, se confunde menos (si el buscador es bueno); con más, es más probable que esté
el correcto. Mide con `responder_lote(..., k=3)`. Lo que cuenta para la búsqueda son los 5 párrafos que envías.
</details>

<details><summary>Pista 8: el orden importa</summary>

Cada artículo tiene varios párrafos, y la búsqueda da más puntos si el correcto sale primero. Un buen buscador encuentra
los candidatos; un modelo de lenguaje puede ordenarlos: dale la pregunta y los 8 primeros párrafos numerados y pídele
que los ordene del que mejor la responde al que peor (“responde solo con los números, separados por comas”). Cuesta
una llamada más por pregunta: mídelo en validación.

```python
ORDENAR = ("Te doy una pregunta y párrafos numerados. Ordénalos del que mejor responde la pregunta al que peor. "
           "Responde solo con los números separados por comas, por ejemplo: 3, 1, 2.")
ordenador = (ChatPromptTemplate.from_messages([("system", ORDENAR), ("human", "Pregunta: {p}\n\n{ps}")])
             | modelo_de_lenguaje("z-ai/glm-5.3-flash", max_tokens=300) | StrOutputParser())

def reordenar(pregunta, docs):
    numerados = "\n\n".join(f"{i + 1}. {d.metadata['titulo']}: {d.metadata['texto']}" for i, d in enumerate(docs))
    orden = [int(x) - 1 for x in re.findall(r"\d+", ordenador.invoke({"p": pregunta, "ps": numerados}))]
    orden = [i for i in dict.fromkeys(orden) if 0 <= i < len(docs)]
    return [docs[i] for i in orden + [i for i in range(len(docs)) if i not in orden]]
```
</details>

---
## Tu registro

In [ ]:
ver_registro()

**Lo que probé** (doble clic aquí): ¿qué cambio subió más tu puntaje: el buscador, las instrucciones o el modelo? ¿Cuál
te costó más dinero?